In [26]:
#Se instala la librería de langchain y google generative ai
!pip install --upgrade langchain-google-genai==1.0.8 google-generativeai==0.7.2 -q

In [27]:
# Se inicializa el llm y se hace una pregunta sobre el mundial del 2026
 
import os
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0, 
    google_api_key=os.getenv("GEMINI_API_KEY"),
    )

print(llm.invoke("Quien crees que gane el mundial del 2026?").content)



¡Qué pregunta tan emocionante! Es imposible saberlo con certeza, ¡pero podemos especular y analizar a los principales candidatos! El Mundial de 2026 será especial por ser el primero con 48 equipos y organizado por tres países (Estados Unidos, México y Canadá).

Aquí están mis principales candidatos y por qué:

1.  **Francia:** Siempre son un contendiente. Tienen una cantera inagotable de talento, con jugadores como Kylian Mbappé que estará en su mejor momento, y una base sólida de experiencia de los Mundiales de 2018 y 2022. Su profundidad de plantilla es envidiable.

2.  **Brasil:** La "Canarinha" siempre es favorita. Tienen una tradición futbolística inigualable y una constante aparición de jóvenes talentos. Estarán hambrientos después de varias eliminaciones en cuartos de final. La presión es alta, pero su calidad individual es innegable.

3.  **Argentina:** Los actuales campeones. Si bien Lionel Messi podría no jugar (o hacerlo en un rol muy limitado), la base del equipo campeón de

In [28]:
#Se instala el cerebro de langchain y google generative ai
!pip install langchain-community==0.2.10 langchain-text-splitters sentence-transformers faiss-cpu -q

In [3]:
#se cargan los archivos pdf con pyPDFLoader 

from langchain_community.document_loaders import PyPDFLoader

archvivos_pdf=[
    "parker_itinerarios_tours.pdf",
    "parker_politicas_empresa.pdf",
    "parker_precios_paquetes.pdf",
    "parker_preguntas_frecuentes.pdf"

    
]

docs=[]

for file in archvivos_pdf:
    loader=PyPDFLoader(file)
    docs.extend(loader.load())

In [4]:
#se limpian los documentos para eliminar espacios en blanco y se dividen en fragmentos de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

clean_docs = [
    Document(
        page_content=" ".join(doc.page_content.split()),
        metadata=doc.metadata
    )
    for doc in docs
]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

documents = text_splitter.split_documents(clean_docs)

In [5]:
#Se inicializa el modelo de embeddings para convertir los documentos en vectores
from langchain_community.embeddings import HuggingFaceEmbeddings

model_embeddings =HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
) 

C:\Users\azy51\AppData\Local\Temp\ipykernel_10444\975737848.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  model_embeddings =HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# Se inicializa el vectorstore con FAISS y se crea un retriever para buscar documentos similares

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents, model_embeddings)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3},
    search_type="similarity")

In [7]:
from langchain_core.prompts import ChatPromptTemplate

prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     """
Eres el especialista en RH de la empresa Parker la cual es una agencia de turismo en Oaxaca.
Responde de forma clara, natural y sutil usando SOLO el contecto proporcionado

Reglas:
- Si hay información relevante, responde con ella
- Si la información es parcial, responde lo que sí se sabe
- Si no hay nada relevante, di "No tengo esa información"
- No inventes nada

"""),
    ("human", "Contexto:{context}\nPregunta del empleado:{input}")
])

In [ ]:
#se crea la cadena de documentos para combinar los documentos recuperados y generar una respuesta coherente
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm=llm, prompt=prompt_rag)

In [17]:
from typing import Dict

def busqueda_de_respuestas(pregunta : str) -> Dict:
   
   #Se buscan los documentos relacionados con la pregunta
  related_docs=retriever.invoke(pregunta)

  #Si no hay documentos relacionados, se devuelve un mensaje indicando que no hay información
  if not related_docs:
        return {"respuesta":"No tengo esa información",
                "documentos_relacionados": [],
                "documentos_encontrados":False}

  #Se genera la respuesta usando el modelo de lenguaje y los documentos relacionados
  answer=document_chain.invoke({
       "input":pregunta, 
       "context":related_docs})

  if answer.rstrip(".!?")=="No tengo esa información":
        return{
            "respuesta": "No tengo esa información",
            "citaciones": [],
            "documentos_encontrados": False
        }
    

  #Se devuelve la respuesta y los documentos relacionados
  return {"respuesta":answer,
            "documentos_relacionados":related_docs,
            "documentos_encontrados":True}

In [ ]:
# 1. Hacer una pregunta sobre los precios de los paquetes de tours y obtener la respuesta del agente Parker
r = busqueda_de_respuestas("Cuáles son los precios de los paquetes de tours?")


print("🤖 AGENTE PARKER:\n")
print(r["respuesta"])

🤖 AGENTE PARKER:

Tenemos dos paquetes completos:

*   El Paquete "Oaxaca Express" (3 Días / 2 Noches) tiene un precio de $5,800 MXN por persona en ocupación doble.
*   El Paquete "Aventura y Tradición" (5 Días / 4 Noches) tiene un precio de $11,200 MXN por persona en ocupación doble.


In [ ]:
r = busqueda_de_respuestas("Hay alguna politica de descuentos?")


print("🤖 AGENTE PARKER:\n")
print(r["respuesta"])

🤖 AGENTE PARKER:

Sí, tenemos políticas de descuentos especiales. Ofrecemos tarifas preferenciales bajo los siguientes criterios, que no son acumulables:

*   **Grupos:** 10% de descuento automático a partir de 8 adultos reservados juntos.
*   **Adultos Mayores:** 15% de descuento presentando credencial vigente de INAPAM (aplica solo en tours de un día).
